# Part 7：Marker-free 週期偵測

## 研究問題

只用五條共同 GPU metric time series，能否在 5–15 秒 prefix 內找出 iteration-like period？

## 目前結論

Adaptive policy：48/48 在 10% period error 內；combined mean APE 1.01%、median 0.81%，平均 emission 10.06 秒。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 匯出 marker-free 1 ms GPU metric time series。
2. 做 late-tail、robust normalization 與多尺度重採樣。
3. 以 ACF peak、prominence 與 channel agreement 產生候選。
4. 僅在 training workloads 選 adaptive threshold。
5. 評估時才讀 oracle marker 計算 period APE。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/marker_free_timeseries_manifest.csv",
    "analysis_cache/marker_free_timeseries",
    "analysis_outputs/iteration_validation.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/extract_marker_free_timeseries.py"],
    ["analysis/detect_marker_free_cycles.py"],
    ["analysis/evaluate_marker_free_cycles.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part7_marker_free_cycles/adaptive_metrics.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
detections = load_csv("analysis_outputs/model_experiments/part7_marker_free_cycles/deployment_oof_detections.csv")
evaluation = load_csv("analysis_outputs/model_experiments/part7_marker_free_cycles/oof_cycle_predictions.csv")
display(detections.head())
display(evaluation.sort_values("period_ape_percent", ascending=False).head(15))

metrics.set_index("device_id")[["mean_period_ape_percent", "median_period_ape_percent", "p90_period_ape_percent"]].plot.bar(
    figsize=(8, 4), ylabel="Period APE (%)", title="Adaptive marker-free period detection"
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part7_marker_free_cycles/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 7 checkpoint: marker-free cycle detection

The detector reads only cached GPU metric time series. It never queries marker ranges. This evaluator reads marker ranges only after detection to create hidden period labels.

Each outer fold holds out the same workload on both GPUs. Detector bin width and tail fraction are selected from the other 23 workloads (46 profiles).

| Horizon | Device | N | Coverage | Mean period APE | Median | P90 | Max | <=5% | <=10% | Half | Double |
|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 5s | RTX4090 | 24 | 100.0% | 93.92% | 31.88% | 269.79% | 286.56% | 50.0% | 50.0% | 0 | 1 |
| 5s | RTX5090 | 24 | 100.0% | 94.97% | 5.55% | 252.18% | 274.35% | 50.0% | 54.2% | 0 | 1 |
| 5s | both_devices | 48 | 100.0% | 94.44% | 5.75% | 262.54% | 286.56% | 50.0% | 52.1% | 0 | 2 |
| 7s | RTX4090 | 24 | 100.0% | 119.23% | 52.90% | 288.25% | 325.21% | 50.0% | 50.0% | 0 | 1 |
| 7s | RTX5090 | 24 | 100.0% | 6.29% | 2.74% | 11.11% | 51.67% | 54.2% | 87.5% | 1 | 0 |
| 7s | both_devices | 48 | 100.0% | 62.76% | 2.74% | 247.65% | 325.21% | 52.1% | 68.8% | 1 | 1 |
| 9s | RTX4090 | 24 | 100.0% | 25.40% | 3.07% | 70.79% | 212.49% | 54.2% | 66.7% | 1 | 1 |
| 9s | RTX5090 | 24 | 100.0% | 1.02% | 0.79% | 1.94% | 3.63% | 100.0% | 100.0% | 0 | 0 |
| 9s | both_devices | 48 | 100.0% | 13.21% | 1.13% | 32.79% | 212.49% | 77.1% | 83.3% | 1 | 1 |
| 12s | RTX4090 | 15 | 100.0% | 1.44% | 1.14% | 3.00% | 4.32% | 100.0% | 100.0% | 0 | 0 |
| 12s | RTX5090 | 14 | 100.0% | 0.60% | 0.45% | 1.40% | 1.58% | 100.0% | 100.0% | 0 | 0 |
| 12s | both_devices | 29 | 100.0% | 1.04% | 0.83% | 2.43% | 4.32% | 100.0% | 100.0% | 0 | 0 |
| 15s | RTX4090 | 13 | 100.0% | 1.21% | 0.83% | 2.69% | 3.15% | 100.0% | 100.0% | 0 | 0 |
| 15s | RTX5090 | 7 | 100.0% | 0.71% | 0.60% | 1.28% | 1.49% | 100.0% | 100.0% | 0 | 0 |
| 15s | both_devices | 20 | 100.0% | 1.03% | 0.69% | 2.39% | 3.15% | 100.0% | 100.0% | 0 | 0 |

## Adaptive marker-free policy (nested threshold selection)

Each held-out workload uses a confidence threshold selected only from the other 23 workloads. The detector tries 9s, then continues to 12s and 15s only if confidence is insufficient.

| Device | Profiles | Accepted | Coverage | Mean APE | Median | P90 | Max | <=10% | Mean emission |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| RTX4090 | 24 | 24 | 100.0% | 1.17% | 0.99% | 2.65% | 5.61% | 100.0% | 10.62s |
| RTX5090 | 24 | 24 | 100.0% | 0.86% | 0.77% | 1.56% | 2.25% | 100.0% | 9.50s |
| both_devices | 48 | 48 | 100.0% | 1.01% | 0.81% | 1.86% | 5.61% | 100.0% | 10.06s |

## Checkpoint decision

A forced period at 5s or 7s is unsafe, and a forced 9s period still has large outliers. The adaptive confidence-gated policy is suitable as a preliminary marker-free segmentation front end on this dataset: all 48 held-out profiles were accepted by 15s and all were within 10% of the hidden marker period.

This is not yet evidence of generalization to arbitrary user workloads. The dataset contains 24 workload configurations on only two GPUs, and the source traces were collected from instrumented runs even though marker/callback fields are never exposed to the detector. The next experiment should build per-cycle features from these inferred periods and evaluate cross-device runtime prediction with the same workload-held-out split.

## Worst out-of-fold detections

| Horizon | Device | Workload | Detected ms | Oracle ms | APE | Class | Configuration |
|---:|---|---|---:|---:|---:|---|---|
| 7s | RTX4090 | C18 | 220.00 | 51.74 | 325.21% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 7s | RTX4090 | C17 | 220.00 | 52.26 | 321.00% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 7s | RTX4090 | C21 | 220.00 | 54.78 | 301.63% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX4090 | C18 | 200.00 | 51.74 | 286.56% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX4090 | C17 | 200.00 | 52.26 | 282.73% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX4090 | C13 | 191.00 | 50.42 | 278.84% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX5090 | C21 | 180.00 | 48.08 | 274.35% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX5090 | C18 | 180.00 | 48.77 | 269.07% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 5s | RTX5090 | C17 | 167.00 | 46.42 | 259.74% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 7s | RTX4090 | C13 | 180.00 | 50.42 | 257.02% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.65}` |
| 5s | RTX4090 | C14 | 180.00 | 51.62 | 248.67% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.5}` |
| 7s | RTX4090 | C14 | 180.00 | 51.62 | 248.67% | other_error | `{"bin_ms": 1.0, "tail_fraction": 0.65}` |

## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。